In [1]:
!pip install moabb mne torch torchvision --quiet

In [2]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery


ds = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[2])



Choosing from all possible events
/Users/temp/anaconda3/envs/robust_eeg/lib/python3.10/site-packages/moabb/datasets/download.py:56: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


MNE_DATA is not already configured. It will be set to default location in the home directory - /Users/temp/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/Users/temp/anaconda3/envs/robust_eeg/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 43.1M/43.1M [00:00<00:00, 36.4GB/s]
SHA256 hash of downloaded file: 5ddd5cb520b1692c3ba1363f48d98f58f0e46f3699ee50d749947950fc39db27
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/Users/temp/anaconda3/envs/robust_eeg/lib/python3.10/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100

In [3]:
class_map = {'left_hand': 0, 'right_hand': 1, 'feet': 2, 'tongue': 3}
y_numeric = [class_map[label] for label in labels]

In [4]:
y_numeric

[0,
 1,
 1,
 0,
 1,
 0,
 1,
 2,
 1,
 3,
 0,
 2,
 1,
 0,
 3,
 3,
 3,
 3,
 3,
 0,
 2,
 1,
 0,
 0,
 2,
 3,
 0,
 2,
 2,
 2,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 2,
 1,
 2,
 2,
 3,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 2,
 1,
 0,
 0,
 1,
 2,
 3,
 1,
 2,
 0,
 0,
 0,
 3,
 1,
 1,
 0,
 0,
 2,
 0,
 1,
 3,
 3,
 2,
 0,
 3,
 3,
 1,
 3,
 3,
 1,
 0,
 1,
 2,
 2,
 2,
 3,
 2,
 0,
 3,
 1,
 2,
 1,
 2,
 3,
 1,
 2,
 0,
 0,
 0,
 3,
 1,
 0,
 2,
 0,
 2,
 1,
 3,
 0,
 2,
 2,
 0,
 2,
 1,
 3,
 3,
 3,
 2,
 0,
 3,
 1,
 3,
 1,
 0,
 2,
 1,
 0,
 2,
 2,
 0,
 2,
 3,
 3,
 1,
 0,
 1,
 3,
 1,
 3,
 2,
 1,
 1,
 1,
 2,
 3,
 0,
 1,
 3,
 0,
 2,
 2,
 3,
 0,
 0,
 2,
 1,
 3,
 3,
 3,
 1,
 0,
 2,
 1,
 3,
 0,
 3,
 2,
 1,
 3,
 3,
 0,
 1,
 1,
 2,
 3,
 1,
 0,
 0,
 3,
 1,
 0,
 2,
 1,
 1,
 2,
 0,
 3,
 2,
 2,
 2,
 2,
 0,
 1,
 0,
 1,
 0,
 0,
 2,
 2,
 1,
 2,
 3,
 0,
 3,
 0,
 0,
 1,
 3,
 2,
 1,
 3,
 2,
 3,
 2,
 3,
 1,
 1,
 3,
 0,
 1,
 1,
 1,
 2,
 3,
 0,
 3,
 0,
 2,
 0,
 3,
 0,
 2,
 0,
 1,
 2,
 2,
 3,
 0,
 1,
 3,
 1,
 2,
 2,
 0,
 3,
 1,
 3,
 0,
 0,
 2,
 2,
 1,


In [5]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y_numeric, dtype=torch.long)

full_dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Define a simple CNN (adjust input dims if needed)
class CNN(nn.Module):
    def __init__(self, input_shape, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)

        # Compute flattened size dynamically
        with torch.no_grad():
            x = torch.zeros(1, 1, *input_shape)
            x = self.pool(F.relu(self.conv1(x)))
            x = self.pool(F.relu(self.conv2(x)))
            self.flattened_size = x.numel()

        self.fc1 = nn.Linear(self.flattened_size, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
    

# Instantiate model, criterion, optimizer
input_shape = X_tensor.shape[2:]  # (channels, times)
num_classes = len(set(y_numeric))  # or total classes you have
model = CNN(input_shape, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")

Epoch 1/20 - Loss: 2.6403 - Accuracy: 0.2569
Epoch 2/20 - Loss: 1.3814 - Accuracy: 0.2656


KeyboardInterrupt: 